# Video 11: End-to-End Drug Discovery Report

**AI for Drug Discovery series | DigitalSreeni**

---

This notebook compiles results from all previous videos into a single structured PDF report.
It loads saved outputs from Google Drive across all videos and produces a professional PDF
documenting the full computational pipeline from target identification through molecular docking.

**Required Drive folder structure:**
- `Video3_molecular_representations/data/egfr_qsar_ready.csv`
- `Video4_qsar_modeling/data/egfr_raw_chembl.csv`, `screening_library_scored.csv`
- `Video5_virtual_screening/data/top20_diverse_hits.csv`
- `Video6_Cellpainting/results/single_cell_features_example.csv`, `well_profile_example.csv`
- `Video7b_admet/results/video7b_admet_results.csv`
- `Video8_generation/results/video8_top_candidates.csv`
- `Video9_llm/results/resistance_landscape.csv`, `scaffold_literature.csv`, `research_briefing.txt`
- `Video10_docking/results/docking_results.csv`, `all_poses.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 1: Installs, Imports, and Configuration

We install `pdfkit` and `wkhtmltopdf` for HTML-to-PDF conversion. wkhtmltopdf is a headless
browser renderer that handles CSS and embedded base64 images correctly, producing a clean PDF
without requiring LaTeX. All other libraries are standard in Colab.

We also run a preflight check on all input file paths so any missing file is flagged immediately.

In [ ]:
# Cell 1 -- installs, imports, configuration

!pip install pdfkit -q
!apt-get install -qq wkhtmltopdf

import os
import warnings
from pathlib import Path
from datetime import date
from io import BytesIO
import base64

import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pdfkit

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'

# Root path for all series data on Google Drive
ROOT = Path('/content/drive/MyDrive/ColabNotebooks/AI_for_drug_discovery')

# Results directory for this notebook
RESULTS_DIR = ROOT / 'Video11_report' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# All input file paths
PATHS = {
    'v3_qsar':       ROOT / 'Video3_molecular_representations' / 'data'    / 'egfr_qsar_ready.csv',
    'v4_chembl':     ROOT / 'Video4_qsar_modeling'             / 'data'    / 'egfr_raw_chembl.csv',
    'v4_screening':  ROOT / 'Video4_qsar_modeling'             / 'data'    / 'screening_library_scored.csv',
    'v5_hits':       ROOT / 'Video5_virtual_screening'         / 'data'    / 'top20_diverse_hits.csv',
    'v6_cells':      ROOT / 'Video6_Cellpainting'              / 'results' / 'single_cell_features_example.csv',
    'v6_wells':      ROOT / 'Video6_Cellpainting'              / 'results' / 'well_profile_example.csv',
    'v7b_admet':     ROOT / 'Video7b_admet'                    / 'results' / 'video7b_admet_results.csv',
    'v8_candidates': ROOT / 'Video8_generation'                / 'results' / 'video8_top_candidates.csv',
    'v9_resistance': ROOT / 'Video9_llm'                       / 'results' / 'resistance_landscape.csv',
    'v9_scaffold':   ROOT / 'Video9_llm'                       / 'results' / 'scaffold_literature.csv',
    'v9_briefing':   ROOT / 'Video9_llm'                       / 'results' / 'research_briefing.txt',
    'v10_docking':   ROOT / 'Video10_docking'                  / 'results' / 'docking_results.csv',
    'v10_poses':     ROOT / 'Video10_docking'                  / 'results' / 'all_poses.csv',
}

# Preflight check
print('File availability check:')
missing = []
for name, path in PATHS.items():
    status = 'FOUND  ' if path.exists() else 'MISSING'
    if not path.exists():
        missing.append(name)
    print(f'  {status}  {name:<18}  {path.name}')

print(f'\nMissing: {len(missing)}')
if missing:
    print('  Upload missing files to Drive before continuing.')
else:
    print('  All files found. Ready to generate report.')
print(f'Output: {RESULTS_DIR}')
print(f'Date:   {date.today()}')

Extracting templates from packages: 100%
Selecting previously unselected package libavahi-core7:amd64.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../00-libavahi-core7_0.8-5ubuntu5.5_amd64.deb ...
Unpacking libavahi-core7:amd64 (0.8-5ubuntu5.5) ...
Selecting previously unselected package libdaemon0:amd64.
Preparing to unpack .../01-libdaemon0_0.14-7.1ubuntu3_amd64.deb ...
Unpacking libdaemon0:amd64 (0.14-7.1ubuntu3) ...
Selecting previously unselected package avahi-daemon.
Preparing to unpack .../02-avahi-daemon_0.8-5ubuntu5.5_amd64.deb ...
Unpacking avahi-daemon (0.8-5ubuntu5.5) ...
Selecting previously unselected package libdouble-conversion3:amd64.
Preparing to unpack .../03-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../04-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (

## Cell 2: Utility Functions

`fig_to_base64` embeds matplotlib figures directly in the HTML as base64 PNG strings,
keeping the final report self-contained with no external image dependencies.
`df_to_html_table` converts a pandas DataFrame to a styled HTML table string.

In [ ]:
# Cell 2 - utility functions

def fig_to_base64(fig):
    """Convert a matplotlib figure to a base64 PNG string for HTML embedding."""
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return f'data:image/png;base64,{encoded}'


def df_to_html_table(df, max_rows=20):
    """Convert a DataFrame to a styled HTML table string."""
    display_df = df.head(max_rows) if len(df) > max_rows else df
    return display_df.to_html(index=False, border=0, classes='data-table')


COLOR_NAVY  = '#1B2A4A'
COLOR_TEAL  = '#0D9488'
COLOR_AMBER = '#F59E0B'
COLOR_RED   = '#DC2626'
COLOR_GRAY  = '#94A3B8'

print('Utility functions defined.')

Utility functions defined.


## Cell 3: Load All Data

We load every CSV and text file saved across the series from Google Drive.
Each dataset is summarized after loading to confirm it looks correct.

In [ ]:
# Cell 3 -- load all data from Google Drive

df_qsar      = pd.read_csv(PATHS['v3_qsar'])
print(f'Video 3  QSAR dataset:       {len(df_qsar):>6} compounds, {len(df_qsar.columns)} columns')

df_chembl    = pd.read_csv(PATHS['v4_chembl'])
df_screening = pd.read_csv(PATHS['v4_screening'])
print(f'Video 4  ChEMBL raw:         {len(df_chembl):>6} compounds')
print(f'Video 4  Screening library:  {len(df_screening):>6} compounds scored')

df_hits = pd.read_csv(PATHS['v5_hits'])
print(f'Video 5  Diverse hits:       {len(df_hits):>6} compounds')

df_cells = pd.read_csv(PATHS['v6_cells'])
df_wells = pd.read_csv(PATHS['v6_wells'])
print(f'Video 6  Single-cell:        {len(df_cells):>6} cells, {df_cells.shape[1]} features')
print(f'Video 6  Well profiles:      {len(df_wells):>6} wells')

df_admet = pd.read_csv(PATHS['v7b_admet'])
print(f'Video 7b ADMET results:      {len(df_admet):>6} compounds')

df_v8 = pd.read_csv(PATHS['v8_candidates'])
print(f'Video 8  Top candidates:     {len(df_v8):>6} compounds')

df_resistance = pd.read_csv(PATHS['v9_resistance'])
df_scaffold   = pd.read_csv(PATHS['v9_scaffold'])
briefing_text = PATHS['v9_briefing'].read_text(encoding='utf-8')
print(f'Video 9  Resistance papers:  {len(df_resistance):>6} abstracts')
print(f'Video 9  Scaffold papers:    {len(df_scaffold):>6} abstracts')

df_docking = pd.read_csv(PATHS['v10_docking'])
df_poses   = pd.read_csv(PATHS['v10_poses'])
print(f'Video 10 Docking results:    {len(df_docking):>6} compounds')

print('\nAll data loaded successfully.')

Video 3  QSAR dataset:          479 compounds, 11 columns
Video 4  ChEMBL raw:          10000 compounds
Video 4  Screening library:    4741 compounds scored
Video 5  Diverse hits:           20 compounds
Video 6  Single-cell:            39 cells, 29 features
Video 6  Well profiles:           1 wells
Video 7b ADMET results:          20 compounds
Video 8  Top candidates:         15 compounds
Video 9  Resistance papers:      20 abstracts
Video 9  Scaffold papers:        10 abstracts
Video 10 Docking results:         3 compounds

All data loaded successfully.


## Cell 4: Generate Report Figures

Four figures are generated and embedded in the report as base64 PNG strings.

- **Figure 1:** Compound funnel showing candidates narrowing at each pipeline stage
- **Figure 2:** ADMET score distribution for all 20 virtual screening hits
- **Figure 3:** Docking score vs ADMET score for the three compounds tested in Video 10
- **Figure 4:** Resistance mutation frequency from Video 9 literature mining

In [ ]:
# Cell 4 - generate all four report figures

from collections import Counter
import ast

figures = {}

# Figure 1: Compound funnel
fig, ax = plt.subplots(figsize=(10, 3.8))
stages = ['ChEMBL\nraw', 'QSAR\ndataset', 'Screening\nlibrary',
          'Top 20\nhits', 'ADMET\nscored', 'Top 3\nleads',
          'Generated\ncandidates', 'Docking\ntested']
counts = [len(df_chembl), len(df_qsar), len(df_screening),
          20, 20, 3, len(df_v8), 3]
bar_c  = [COLOR_TEAL if c > 500 else COLOR_AMBER if c > 10 else COLOR_NAVY for c in counts]
bars = ax.bar(stages, counts, color=bar_c, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Number of compounds', fontsize=11)
ax.set_title('Compound Funnel: ChEMBL to Docking Candidates', fontsize=12, fontweight='bold')
ax.set_yscale('log')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
            f'{count:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(fontsize=9)
plt.tight_layout()
figures['funnel'] = fig_to_base64(fig)
print('Figure 1: compound funnel done')

# Figure 2: ADMET scores
fig, ax = plt.subplots(figsize=(9, 5))
admet_s = df_admet.sort_values('admet_score', ascending=True)
bc2 = [COLOR_TEAL if s >= 7 else COLOR_AMBER if s >= 5 else COLOR_GRAY
       for s in admet_s['admet_score']]
bars = ax.barh(admet_s['compound_id'], admet_s['admet_score'], color=bc2, edgecolor='white')
ax.axvline(x=7, color=COLOR_NAVY, linestyle='--', linewidth=1.5, alpha=0.8, label='Lead threshold (7)')
ax.set_xlabel('ADMET Score (out of 10)', fontsize=11)
ax.set_title('ADMET Scores: All 20 Virtual Screening Hits', fontsize=12, fontweight='bold')
ax.set_xlim(0, 10)
ax.legend(fontsize=9)
for bar, score in zip(bars, admet_s['admet_score']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(score), va='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
figures['admet'] = fig_to_base64(fig)
print('Figure 2: ADMET scores done')

# Figure 3: Docking vs ADMET
needed = ['Compound', 'ADMET Score', 'Docking (kcal/mol)']
if all(c in df_docking.columns for c in needed):
    df_dp = df_docking[needed].dropna()
    dc = [COLOR_NAVY, COLOR_TEAL, COLOR_AMBER][:len(df_dp)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].barh(df_dp['Compound'], df_dp['Docking (kcal/mol)'], color=dc, edgecolor='white')
    axes[0].axvline(x=-7, color=COLOR_RED, linestyle='--', linewidth=1.2, alpha=0.7, label='-7 threshold')
    axes[0].set_xlabel('Docking score (kcal/mol)', fontsize=10)
    axes[0].set_title('Predicted Binding Affinity', fontsize=11, fontweight='bold')
    axes[0].legend(fontsize=8)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    axes[1].barh(df_dp['Compound'], df_dp['ADMET Score'], color=dc, edgecolor='white')
    axes[1].set_xlabel('ADMET score (out of 10)', fontsize=10)
    axes[1].set_title('ADMET Score', fontsize=11, fontweight='bold')
    axes[1].set_xlim(0, 10)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    plt.suptitle('Final Candidates: Docking vs ADMET', fontsize=12, fontweight='bold')
    plt.tight_layout()
    figures['docking'] = fig_to_base64(fig)
    print('Figure 3: docking vs ADMET done')
else:
    figures['docking'] = None
    print(f'Figure 3 skipped. Columns found: {list(df_docking.columns)}')

# Figure 4: Resistance mutations
all_muts = []
for val in df_resistance['resistance_mutations'].dropna():
    try:
        muts = ast.literal_eval(str(val))
        if isinstance(muts, list):
            all_muts.extend(muts)
    except Exception:
        pass

if all_muts:
    mc = Counter(all_muts).most_common(8)
    lbs, cts = zip(*mc)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(lbs[::-1], cts[::-1], color=COLOR_TEAL, edgecolor='white')
    ax.set_xlabel('Papers mentioning mutation', fontsize=10)
    ax.set_title('EGFR Resistance Mutations (20 PubMed abstracts)', fontsize=11, fontweight='bold')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    figures['mutations'] = fig_to_base64(fig)
    print('Figure 4: resistance mutations done')
else:
    figures['mutations'] = None
    print('Figure 4 skipped: no mutation data')

print(f'\nTotal figures: {sum(v is not None for v in figures.values())}/{len(figures)}')

Figure 1: compound funnel done
Figure 2: ADMET scores done
Figure 3: docking vs ADMET done
Figure 4: resistance mutations done

Total figures: 4/4


## Cell 5: Build the HTML Report

We assemble all sections into a single styled HTML document. The structure follows a scientific
methods paper with eight sections, one per pipeline stage. All figures are embedded as base64
PNG strings so the HTML file is fully self-contained.

In [ ]:
# Cell 5 - build the full HTML report

from datetime import date
today = date.today().strftime('%B %d, %Y')

def embed_fig(key, caption):
    if figures.get(key):
        src = figures[key]
        return f'<div class="figure-wrap"><img src="{src}"><p class="figure-caption">{caption}</p></div>'
    return ''

# CSS is loaded from a pre-written string to avoid Python parsing conflicts with CSS syntax
CSS = (
    '* { box-sizing: border-box; margin: 0; padding: 0; }\n'
    'body { font-family: Arial, sans-serif; font-size: 11pt; color: #1e293b; background: white; }\n'
    '.cover { background: #1B2A4A; color: white; padding: 60px 50px; page-break-after: always; }\n'
    '.cover h1 { font-size: 26pt; font-weight: bold; margin-bottom: 10px; line-height: 1.2; }\n'
    '.cover .subtitle { font-size: 13pt; color: #0D9488; margin-bottom: 20px; }\n'
    '.cover .meta { font-size: 10pt; color: #94A3B8; margin-top: 6px; }\n'
    '.cover .series-info { border-top: 2px solid #0D9488; margin-top: 28px; padding-top: 16px; }\n'
    '.cover .series-info p { font-size: 10pt; color: #CBD5E1; line-height: 1.7; }\n'
    '.section { padding: 24px 40px; border-bottom: 1px solid #E2E8F0; }\n'
    '.section-header { background: #1B2A4A; color: white; padding: 9px 16px; margin: -24px -40px 18px -40px; }\n'
    '.section-header h2 { font-size: 13pt; font-weight: bold; display: inline; }\n'
    '.section-header .vtag { font-size: 9pt; color: #0D9488; margin-left: 12px; }\n'
    '.sbox { background: #F0FDFA; border-left: 4px solid #0D9488; padding: 12px 16px; margin: 14px 0; }\n'
    '.sbox p { font-size: 10.5pt; line-height: 1.65; }\n'
    '.rbox { background: #FFFBEB; border-left: 4px solid #F59E0B; padding: 11px 16px; margin: 12px 0; }\n'
    '.rbox p { font-size: 10.5pt; line-height: 1.65; }\n'
    '.mrow { display: flex; gap: 14px; margin: 16px 0; flex-wrap: wrap; }\n'
    '.mc { background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 5px; padding: 12px 16px; flex: 1; min-width: 130px; }\n'
    '.mc .lb { font-size: 8.5pt; color: #64748B; text-transform: uppercase; margin-bottom: 3px; }\n'
    '.mc .vl { font-size: 16pt; font-weight: bold; color: #1B2A4A; }\n'
    '.mc .un { font-size: 8.5pt; color: #94A3B8; }\n'
    '.data-table { width: 100%; border-collapse: collapse; font-size: 8.5pt; margin: 12px 0; }\n'
    '.data-table th { background: #1B2A4A; color: white; padding: 7px 9px; text-align: left; font-weight: bold; }\n'
    '.data-table td { padding: 5px 9px; border-bottom: 1px solid #E2E8F0; }\n'
    '.data-table tr:nth-child(even) td { background: #F8FAFC; }\n'
    '.fw { text-align: center; margin: 18px 0; }\n'
    '.fw img { max-width: 100%; border: 1px solid #E2E8F0; }\n'
    '.fc { font-size: 8.5pt; color: #64748B; margin-top: 5px; font-style: italic; }\n'
    '.pt { width: 100%; border-collapse: collapse; margin: 16px 0; }\n'
    '.pt th { background: #243352; color: white; padding: 8px 11px; font-size: 10pt; text-align: left; }\n'
    '.pt td { padding: 7px 11px; font-size: 10pt; border-bottom: 1px solid #E2E8F0; vertical-align: top; }\n'
    '.pt tr:nth-child(even) td { background: #F8FAFC; }\n'
    '.tag { background: #0D9488; color: white; padding: 2px 7px; border-radius: 9px; font-size: 8pt; font-weight: bold; }\n'
    '.rc { background: #1B2A4A; color: white; border-radius: 5px; padding: 18px 22px; margin: 12px 0; }\n'
    '.rc h3 { font-size: 12pt; color: #0D9488; margin-bottom: 9px; }\n'
    '.rc p { font-size: 10pt; line-height: 1.7; color: #CBD5E1; }\n'
    '.sc { text-align: center; display: inline-block; margin-right: 22px; margin-top: 12px; }\n'
    '.sc .sl { font-size: 8pt; color: #94A3B8; display: block; }\n'
    '.sc .sv { font-size: 15pt; font-weight: bold; color: #F59E0B; display: block; }\n'
    '.ft { background: #F8FAFC; border-top: 2px solid #1B2A4A; padding: 16px 40px; margin-top: 24px; }\n'
    '.ft p { font-size: 9pt; color: #64748B; line-height: 1.6; }\n'
    'h3.sub { font-size: 11pt; color: #1B2A4A; margin: 14px 0 8px; }\n'
    'p.body { font-size: 10.5pt; line-height: 1.7; color: #334155; margin-top: 8px; }\n'
)

# Pipeline overview table
pr = [
    ('1',  'Target Identification',      'EGFR in NSCLC',                              'Biological context and target rationale',           'Video 1'),
    ('2',  'Dose-Response Modeling',     'Synthetic IC50 data',                         'Hill equation fitting demonstrated',                'Video 2'),
    ('3',  'Molecular Representations',  f'{len(df_qsar):,} compounds',                  'QSAR-ready feature table built from ChEMBL',        'Video 3'),
    ('4',  'QSAR Modeling',              f'{len(df_chembl):,} ChEMBL compounds',          'XGBoost classifier: ROC-AUC 0.890, Accuracy 83%',   'Video 4'),
    ('5',  'Virtual Screening',          f'{len(df_screening):,} compounds screened',    '20 diverse hits via MaxMin algorithm',              'Video 5'),
    ('6',  'Cell Painting (MoA)',        '20 hits profiled',                            'Hits confirmed vs EGFR inhibitor references',       'Video 6'),
    ('7',  'ADMET Prediction',           '20 hits scored',                              '3 leads at 7/10: CHEMBL382797, 13462, 112582',      'Video 7b'),
    ('8',  'Molecular Generation',       'Scaffold decoration + SelfiesGPT',            'Best: 9/10 (GPT), 8/10 (decoration)',               'Video 8'),
    ('9',  'LLM Literature Mining',      '30 PubMed abstracts',                         'T790M, C797S as dominant resistance mutations',     'Video 9'),
    ('10', 'Molecular Docking',          '3 compounds vs PDB 1IEP',                     'CHEMBL382797: -7.882 kcal/mol best score',          'Video 10'),
]
pt = '<table class="pt"><tr><th>#</th><th>Stage</th><th>Input</th><th>Key Result</th><th>Video</th></tr>'
for r in pr:
    pt += f'<tr><td>{r[0]}</td><td><b>{r[1]}</b></td><td>{r[2]}</td><td>{r[3]}</td><td><span class="tag">{r[4]}</span></td></tr>'
pt += '</table>'

# ADMET table
ac = ['compound_id','admet_score','sol_flag','lipo_flag','bbb_flag','clintox_flag']
admet_t = df_to_html_table(df_admet[ac].rename(columns={
    'compound_id':'Compound','admet_score':'ADMET','sol_flag':'Solubility',
    'lipo_flag':'Lipophilicity','bbb_flag':'BBB','clintox_flag':'ClinTox'}))

# Video 8 table
v8_t = df_to_html_table(df_v8[['smiles','admet_score','method']].rename(columns={
    'smiles':'SMILES','admet_score':'ADMET Score','method':'Method'}))

# Docking table
dock_t = df_to_html_table(df_docking) if not df_docking.empty else '<p>No docking data.</p>'

# Resistance key findings
rf = df_resistance['key_finding'].dropna().tolist()[:5] if 'key_finding' in df_resistance.columns else []
fh = ''.join([f'<li style="margin-bottom:5px;font-size:10pt;">{f}</li>' for f in rf])

# Briefing excerpt
bl = [l for l in briefing_text.split('\n') if l.strip() and not l.startswith('=')]
be = ' '.join(bl[:6])[:700] + '...'

# Assemble HTML sections
cover = f'''
<div class="cover">
  <h1>Computational Drug Discovery Report</h1>
  <p class="subtitle">EGFR-Targeted Inhibitor Discovery: End-to-End Pipeline Summary</p>
  <p class="meta">Generated: {today}</p>
  <p class="meta">Author: Sreenivas Bhattiprolu | DigitalSreeni</p>
  <p class="meta">Series: AI for Drug Discovery | Videos 1 through 10</p>
  <p class="meta">Code: https://github.com/bnsreenu/AI-for-Drug-Discovery</p>
  <div class="series-info">
    <p>This report documents the complete computational pipeline developed across the AI for Drug
    Discovery tutorial series. Starting from target identification, the pipeline progresses through
    hit discovery, QSAR modeling, virtual screening, morphological profiling, ADMET prediction,
    molecular generation, literature mining, and molecular docking. All analyses used EGFR
    (Epidermal Growth Factor Receptor) as the target in non-small cell lung cancer.</p>
  </div>
</div>'''

s1 = f'''
<div class="section">
  <div class="section-header"><h2>Pipeline Overview</h2></div>
  <div class="sbox"><p>The pipeline follows the standard early drug discovery workflow.
  Each stage builds on the previous: hits from virtual screening feed into ADMET prediction,
  which feeds into molecular generation, which feeds into docking. Literature mining provides
  biological context at the transition to experimental work.</p></div>
  {pt}
  {embed_fig('funnel', 'Figure 1. Compound funnel showing how candidates narrow at each pipeline stage.')}
</div>'''

s2 = f'''
<div class="section">
  <div class="section-header"><h2>Target and Dataset <span class="vtag">Videos 1, 3, 4</span></h2></div>
  <div class="sbox"><p>The target is EGFR (Epidermal Growth Factor Receptor), a receptor tyrosine
  kinase mutated or overexpressed in approximately 15% of non-small cell lung cancer cases in
  Western populations and up to 50% in East Asian populations. EGFR drives tumor proliferation
  through constitutive activation of downstream signaling pathways. Three generations of approved
  EGFR inhibitors exist, with resistance remaining the primary clinical challenge.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">ChEMBL compounds</div><div class="vl">{len(df_chembl):,}</div><div class="un">EGFR bioactivity records</div></div>
    <div class="mc"><div class="lb">QSAR features</div><div class="vl">{len(df_qsar.columns)-1}</div><div class="un">Morgan fingerprint bits</div></div>
    <div class="mc"><div class="lb">Actives</div><div class="vl">3,480</div><div class="un">IC50 less than 1 uM</div></div>
    <div class="mc"><div class="lb">Inactives</div><div class="vl">1,699</div><div class="un">IC50 1 uM or greater</div></div>
  </div>
</div>'''

s3 = f'''
<div class="section">
  <div class="section-header"><h2>QSAR Modeling and Virtual Screening <span class="vtag">Videos 4, 5</span></h2></div>
  <div class="sbox"><p>An XGBoost classifier was trained on Morgan fingerprints to predict EGFR
  bioactivity. The model was applied to a screening library of {len(df_screening):,} compounds.
  The top-scoring compounds were filtered for chemical diversity using the MaxMin algorithm,
  selecting 20 compounds that maximize coverage of chemical space while maintaining high
  predicted activity.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">Model</div><div class="vl">XGBoost</div><div class="un">Classifier</div></div>
    <div class="mc"><div class="lb">ROC-AUC</div><div class="vl">0.890</div><div class="un">Test set</div></div>
    <div class="mc"><div class="lb">Accuracy</div><div class="vl">83%</div><div class="un">Test set</div></div>
    <div class="mc"><div class="lb">Diverse hits</div><div class="vl">20</div><div class="un">MaxMin selected</div></div>
  </div>
  <div class="rbox"><p><b>Performance:</b> Precision 0.90, Recall 0.84, F1 0.87 for the active class.
  The model was weighted to favor recall on actives (scale_pos_weight 0.49) to reduce false negatives
  in the screening context.</p></div>
</div>'''

s4 = f'''
<div class="section">
  <div class="section-header"><h2>Cell Painting: Morphological Profiling <span class="vtag">Video 6</span></h2></div>
  <div class="sbox"><p>Cell Painting is a high-content imaging assay where cells are stained with
  six fluorescent dyes labeling different cellular compartments. Each compound produces a unique
  morphological fingerprint. Compounds with similar mechanisms of action cluster together in
  this feature space, allowing hit confirmation against known EGFR inhibitor reference profiles.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">Cells profiled</div><div class="vl">{len(df_cells):,}</div><div class="un">Single-cell measurements</div></div>
    <div class="mc"><div class="lb">Features per cell</div><div class="vl">{df_cells.shape[1]}</div><div class="un">Morphological descriptors</div></div>
    <div class="mc"><div class="lb">Well profiles</div><div class="vl">{len(df_wells)}</div><div class="un">Aggregated profiles</div></div>
    <div class="mc"><div class="lb">Dataset</div><div class="vl">JUMP-CP</div><div class="un">Pilot dataset</div></div>
  </div>
  <div class="rbox"><p><b>Result:</b> The 20 virtual screening hits were profiled and their
  well-level morphological signatures compared against a panel of known EGFR inhibitors using
  UMAP clustering. Hits confirmed to cluster with the EGFR inhibitor reference group were
  carried forward to ADMET prediction.</p></div>
</div>'''

s5 = f'''
<div class="section">
  <div class="section-header"><h2>ADMET Prediction and Lead Selection <span class="vtag">Video 7b</span></h2></div>
  <div class="sbox"><p>Five XGBoost models were trained on public MoleculeNet datasets to predict
  ADMET properties for each of the 20 hits. Each compound received a composite score out of 10
  based on solubility (ESOL), lipophilicity, blood-brain barrier exclusion (BBBP), clinical toxicity
  risk (ClinTox), and a four-endpoint Tox21 panel. Three compounds scoring 7/10 were selected as
  leads.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">ESOL solubility</div><div class="vl">R2 0.722</div><div class="un">RMSE 1.146</div></div>
    <div class="mc"><div class="lb">Lipophilicity</div><div class="vl">R2 0.519</div><div class="un">RMSE 0.843</div></div>
    <div class="mc"><div class="lb">BBBP</div><div class="vl">AUC 0.927</div><div class="un">Accuracy 89%</div></div>
    <div class="mc"><div class="lb">ClinTox</div><div class="vl">AUC 0.899</div><div class="un">Accuracy 93%</div></div>
  </div>
  {embed_fig('admet', 'Figure 2. ADMET scores for all 20 virtual screening hits. Teal bars indicate leads (score 7 or above).')}
  <h3 class="sub">ADMET Results: All Hits</h3>
  {admet_t}
  <div class="rbox"><p><b>Selected leads:</b> CHEMBL382797 (best BBB exclusion, moderate ClinTox),
  CHEMBL13462 (best Tox21 profile), CHEMBL112582 (lowest ClinTox at 0.17). All three are
  quinazoline scaffold compounds.</p></div>
</div>'''

s6 = f'''
<div class="section">
  <div class="section-header"><h2>Molecular Generation <span class="vtag">Video 8</span></h2></div>
  <div class="sbox"><p>Two generative approaches were applied to CHEMBL382797. Scaffold decoration
  systematically replaced substituents on the quinazoline core with curated fragments targeting
  improved solubility and BBB exclusion. SelfiesGPT, a six-layer transformer trained on 1,472
  ChEMBL EGFR inhibitors in SELFIES representation, generated 500 novel molecules with 100%
  chemical validity.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">Training set</div><div class="vl">1,472</div><div class="un">EGFR inhibitors</div></div>
    <div class="mc"><div class="lb">SELFIES validity</div><div class="vl">100%</div><div class="un">500 generated</div></div>
    <div class="mc"><div class="lb">Best scaffold</div><div class="vl">8/10</div><div class="un">ADMET score</div></div>
    <div class="mc"><div class="lb">Best GPT</div><div class="vl">9/10</div><div class="un">ADMET score</div></div>
  </div>
  <h3 class="sub">Top Generated Candidates</h3>
  {v8_t}
  <div class="rbox"><p><b>Key finding:</b> 139 SelfiesGPT molecules improved on the lead ADMET score.
  The best scaffold-decorated molecule (Scaffold_dec_best) was carried forward to docking.
  </p></div>
</div>'''

s7 = f'''
<div class="section">
  <div class="section-header"><h2>LLM Literature Mining <span class="vtag">Video 9</span></h2></div>
  <div class="sbox"><p>A structured pipeline using the NCBI Entrez API for PubMed access and
  GPT-4o for structured extraction mined 30 abstracts: 20 on EGFR inhibitor resistance and 10
  on quinazoline scaffold medicinal chemistry. Key findings were aggregated and synthesized into
  a research briefing.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">Resistance papers</div><div class="vl">20</div><div class="un">PubMed abstracts</div></div>
    <div class="mc"><div class="lb">Scaffold papers</div><div class="vl">10</div><div class="un">PubMed abstracts</div></div>
    <div class="mc"><div class="lb">Key mutations</div><div class="vl">T790M, C797S</div><div class="un">Most frequent</div></div>
    <div class="mc"><div class="lb">Scaffold confirmed</div><div class="vl">9/10</div><div class="un">Quinazoline papers</div></div>
  </div>
  {embed_fig('mutations', 'Figure 3. Resistance mutations mentioned across 20 PubMed abstracts.')}
  <h3 class="sub">Key Findings from Resistance Literature</h3>
  <ul style="padding-left:20px;margin:10px 0;">{fh}</ul>
  <div class="rbox"><p><b>Briefing excerpt:</b> {be}</p></div>
</div>'''

s8 = f'''
<div class="section">
  <div class="section-header"><h2>Molecular Docking <span class="vtag">Video 10</span></h2></div>
  <div class="sbox"><p>Three candidate molecules were docked against the EGFR kinase domain
  crystal structure (PDB: 1IEP, co-crystallized with erlotinib) using AutoDock Vina v1.2.5.
  The binding site was centered on the ATP binding pocket. Five poses were generated per
  compound at exhaustiveness 4.</p></div>
  <div class="mrow">
    <div class="mc"><div class="lb">Docking engine</div><div class="vl">Vina 1.2.5</div><div class="un">AutoDock Vina</div></div>
    <div class="mc"><div class="lb">Reference structure</div><div class="vl">1IEP</div><div class="un">EGFR + erlotinib</div></div>
    <div class="mc"><div class="lb">Best score</div><div class="vl">-7.882</div><div class="un">kcal/mol</div></div>
    <div class="mc"><div class="lb">Compounds docked</div><div class="vl">3 / 3</div><div class="un">All successful</div></div>
  </div>
  {dock_t}
  {embed_fig('docking', 'Figure 4. Docking scores vs ADMET scores for the three final candidates.')}
  <div class="rbox"><p><b>Key finding:</b> Scaffold_dec_best matched the original lead in docking
  score (-7.869 vs -7.882 kcal/mol) while improving ADMET by one point. SelfiesGPT_rank1 had
  the best ADMET (9/10) but weakest docking (-6.592 kcal/mol), showing that ADMET optimization
  alone does not guarantee target-specific binding.</p></div>
</div>'''

s9 = f'''
<div class="section">
  <div class="section-header"><h2>Integrated Summary and Candidate Recommendation</h2></div>
  <div class="sbox"><p>The recommendation below integrates evidence from all computational stages.
  No single model is treated as definitive. Compounds are evaluated across QSAR prediction,
  ADMET profiling, and molecular docking.</p></div>
  <div class="rc">
    <h3>Primary Recommendation: Scaffold_dec_best</h3>
    <p>The scaffold-decorated variant of CHEMBL382797 with a para-hydroxyphenyl group is the
    strongest overall candidate. It improved ADMET while maintaining identical docking performance.
    The quinazoline scaffold is clinically validated with erlotinib and gefitinib as approved drugs.
    </p>
    <div><span class="sc"><span class="sl">ADMET Score</span><span class="sv">8/10</span></span>
    <span class="sc"><span class="sl">Docking</span><span class="sv">-7.869</span></span>
    <span class="sc"><span class="sl">Scaffold</span><span class="sv">Quinazoline</span></span></div>
  </div>
  <div class="rc" style="background:#243352;margin-top:12px;">
    <h3>Secondary Recommendation: CHEMBL382797</h3>
    <p>The original lead has the best docking score (-7.882 kcal/mol), strong BBB exclusion,
    and moderate ClinTox. As a ChEMBL compound it has additional experimental data available
    and serves as a useful reference for follow-on assays.</p>
    <div><span class="sc"><span class="sl">ADMET Score</span><span class="sv">7/10</span></span>
    <span class="sc"><span class="sl">Docking</span><span class="sv">-7.882</span></span>
    <span class="sc"><span class="sl">Scaffold</span><span class="sv">Quinazoline</span></span></div>
  </div>
  <h3 class="sub">Limitations and Next Steps</h3>
  <p class="body">All results are computational predictions. ADMET models were trained on public
  datasets. Docking used a rigid receptor at exhaustiveness 4 (tutorial setting). LLM synthesis
  contained a factual error identified during review, illustrating that AI-generated content
  requires expert review before informing decisions.</p>
  <p class="body">Recommended next steps: (1) Experimental binding assay (SPR or ITC) for
  Scaffold_dec_best and CHEMBL382797. (2) Repeat docking at exhaustiveness 16+ against both
  wild-type and T790M mutant EGFR (PDB: 2JIU). (3) Cellular activity assay in PC-9 or H1975</p>
</div>'''

footer = f'''
<div class="ft">
  <p><b>AI for Drug Discovery series</b> | Sreenivas Bhattiprolu | DigitalSreeni</p>
  <p>Code: https://github.com/bnsreenu/AI-for-Drug-Discovery</p>
  <p>Generated: {today} | Tools: RDKit, XGBoost, AutoDock Vina, GPT-4o, Biopython, pdfkit</p>
  <p style="margin-top:5px;color:#94A3B8;font-size:8pt;">All predictions are computational and
  require experimental validation before informing any biological or clinical decision.</p>
</div>'''

html_report = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<title>Computational Drug Discovery Report</title>
<style>{CSS}</style></head>
<body>{cover}{s1}{s2}{s3}{s4}{s5}{s6}{s7}{s8}{s9}{footer}</body></html>'''

html_path = RESULTS_DIR / 'egfr_pipeline_report.html'
html_path.write_text(html_report, encoding='utf-8')
print(f'HTML saved: {html_path}')
print(f'HTML size:  {html_path.stat().st_size:,} bytes')

HTML saved: /content/drive/MyDrive/ColabNotebooks/AI_for_drug_discovery/Video11_report/results/egfr_pipeline_report.html
HTML size:  329,821 bytes


## Cell 6: Convert HTML to PDF

We use pdfkit with wkhtmltopdf to convert the self-contained HTML report to PDF.
wkhtmltopdf renders HTML as a headless browser, handling all CSS and embedded images correctly.

In [ ]:
# Cell 6 -- convert HTML to PDF using pdfkit + wkhtmltopdf

import pdfkit

pdf_path = RESULTS_DIR / 'egfr_pipeline_report.pdf'

options = {
    'page-size':   'A4',
    'margin-top':  '15mm',
    'margin-right':'15mm',
    'margin-bottom':'15mm',
    'margin-left': '15mm',
    'encoding':    'UTF-8',
    'enable-local-file-access': None,
    'no-outline':  None,
    'quiet':       '',
}

print('Converting HTML to PDF...')
pdfkit.from_file(str(html_path), str(pdf_path), options=options)

print(f'PDF saved: {pdf_path}')
print(f'PDF size:  {pdf_path.stat().st_size:,} bytes')
print('\nAll outputs in results/:')
for f in sorted(RESULTS_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

## Summary

This notebook compiled results from all ten videos in the AI for Drug Discovery series into
a single structured PDF report.

The report documents the full pipeline: over 12,000 ChEMBL compounds filtered to a QSAR dataset,
an XGBoost model achieving ROC-AUC 0.890, virtual screening to 20 diverse hits, Cell Painting
morphological profiling confirming EGFR activity, ADMET scoring selecting three leads, molecular
generation producing candidates up to 9/10 ADMET, literature mining identifying T790M and C797S
as key resistance mutations, and molecular docking confirming Scaffold_dec_best as the strongest
candidate with docking score -7.869 kcal/mol and ADMET score 8/10.

Both the HTML and PDF versions are saved to `Video11_report/results/` on your Google Drive.